In [ ]:
import pandas as pd
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

data_path = "/Users/bonsitukebeto/Library/CloudStorage/OneDrive-SharedLibraries-NorthwesternUniversity/Arvind Krishna - Data/CAR_-_EP_Flow_Activity_Queue__Agent_Names"
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))

df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 
                                'Activity Start Timestamp', 'Queue Name', 'Agent Name'])

i = 0
for f in files:
    i += 1
    try:
        if f.suffix.lower() == ".csv":
            df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
        else: 
            df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
        
        df_main = pd.concat([df_main, df], ignore_index=True)
    except Exception as e:
        pass

df_main["Activity Start Timestamp"] = pd.to_datetime(df_main["Activity Start Timestamp"], errors='coerce')
df_main = df_main.sort_values(['Contact Session ID', 'Activity Start Timestamp'])

family_sessions = df_main[
    df_main['Flow Name'].str.contains('LegalFamilyMenu', case=False, na=False)
]['Contact Session ID'].unique()

family_df = df_main[df_main['Contact Session ID'].isin(family_sessions)].copy()

family_df['Year'] = family_df['Activity Start Timestamp'].dt.year
family_df['Month_Num'] = family_df['Activity Start Timestamp'].dt.month
family_df['Month_Year'] = family_df['Activity Start Timestamp'].dt.strftime('%b %Y')
family_df['YearMonth'] = family_df['Activity Start Timestamp'].dt.to_period('M').astype(str)

def fiscal_sort(month_num):
    if month_num >= 10:  
        return month_num - 9
    else:  
        return month_num + 3

family_df['Sort_Order'] = family_df['Month_Num'].apply(fiscal_sort)
family_df['Month_Display'] = family_df['Sort_Order'].astype(str).str.zfill(2) + '-' + family_df['Month_Year']

def identify_family_submenu(row):
    activity = str(row['Activity Name'])
    queue = str(row['Queue Name'])
    
    if any(menu in activity for menu in ['DivorceOrParentingMenu', 'SimpleDivorceMenu', 'ChildSupportMenu']):
        return 'Divorce or Parenting'
    
    if 'Education' in queue or 'Education' in activity:
        return 'Education'
    
    if 'Family' in queue:
        return 'General Family Issues'
    
    return None

family_df['Submenu'] = family_df.apply(identify_family_submenu, axis=1)

def assign_2level_hierarchy(submenu):
    if submenu is None:
        return {'Level_1': None, 'Level_2': None}
    
    if submenu in ['Divorce or Parenting', 'Education', 'General Family Issues']:
        return {
            'Level_1': 'Family Menu',
            'Level_2': submenu
        }
    else:
        return {
            'Level_1': 'Family Menu',
            'Level_2': None
        }

hierarchy = family_df['Submenu'].apply(assign_2level_hierarchy)
family_df['Level_1'] = hierarchy.apply(lambda x: x['Level_1'])
family_df['Level_2'] = hierarchy.apply(lambda x: x['Level_2'])

session_submenus = family_df[family_df['Submenu'].notna()].groupby(['Contact Session ID', 'Submenu']).agg({
    'Activity Start Timestamp': 'min',
    'Year': 'first',
    'Month_Display': 'first',
    'Month_Year': 'first',
    'Sort_Order': 'first',
    'YearMonth': 'first',
    'Level_1': 'first',
    'Level_2': 'first'
}).reset_index()

time_hierarchy = session_submenus.groupby([
    'Sort_Order',
    'Month_Display',
    'Year',
    'Month_Year',
    'YearMonth',
    'Level_1',
    'Level_2'
]).agg({
    'Contact Session ID': 'nunique'
}).reset_index()

time_hierarchy.columns = [
    'Sort_Order', 'Month_Display', 'Year', 'Month_Year', 'YearMonth',
    'Level_1', 'Level_2', 'Call_Count'
]

time_hierarchy = time_hierarchy.sort_values('Sort_Order')

time_hierarchy['Level_1'] = time_hierarchy['Level_1'].fillna('')
time_hierarchy['Level_2'] = time_hierarchy['Level_2'].fillna('')

if len(time_hierarchy) > 0:
    time_hierarchy.to_csv('family_menu_monthly_drillthrough.csv', index=False)